# Tuotantolinjan anturien redundanssin vähentäminen PROC GVARCLUS -menetelmällä

## Yhteenveto

Monivyöhykkeinen valmistuslinja striimaa kymmeniä korreloituneita anturilukemia, joista monet kantavat samaa taustalla olevaa signaalia. Tämä muistikirja käyttää **PROC GVARCLUS**-menetelmää (muuttujaklusterointi) ryhmittelemään 13 prosessianturia neljään erilliseen klusteriin, ja lukee sitten kunkin klusterin selitysasteet (R²) valitakseen yhden edustavan mittarin klusteria kohden — pienentäen SPC-seurannan laajuutta menettämättä prosessi-informaatiota. Kolme neljästä klusterista vastaa siististi fyysisiä vyöhykkeitä (keskimääräinen R² 0,92, 0,93 ja 0,96); neljäs on yksikanavainen klusteri, jonka menetelmä erotti omakseen ja jota tarkastelemme sivuuttamatta sitä.

## Tietolähteet

Kaikki data luodaan sisäisesti komennoilla `call streaminit(20260531)` ja `rand()` — ei ulkoisia tai verkosta tulevia syötteitä.

| Aineisto | Rivit | Avainmuuttujat | Kuvaus |
|---------|------|---------------|-------------|
| `process_sensors` | 100 | `zone1_a`–`zone1_c`, `zone2_a`–`zone2_c`, `zone3_a`, `zone3_b`, `temp_in`, `temp_out`, `pressure_a`, `pressure_b`, `vibration` | Synteettisiä lukemia 3-vyöhykkeiseltä tuotantolinjalta. Saman vyöhykkeen anturit jakavat piilevän ajurin (korkea korrelaatio); vyöhykkeiden väliset mittarit (lämpötilat sidottu vyöhykkeeseen 1, paineet vyöhykkeeseen 2, tärinä vyöhykkeeseen 3) on lisätty luomaan realistista redundanssia. DATA-vaiheen silmukka pyytää 300 kierrosta, mutta tämä arviointiympäristö toimii ilman lisenssiä ja säilyttää ensimmäiset 100 havaintoa — riittävästi klusterirakenteen palauttamiseksi siististi. |
| `clusters` (OUT=) | 13 | `Variable`, `Cluster`, `RSq_Own` | Yksi rivi jokaista syöttöanturia kohden: klusteri, johon se on osoitettu, ja sen R² oman klusterinsa komponentin kanssa. Tuotettu `OUTPUT OUT=`-lauseella. |

# Tuotantolinjan anturien redundanssin vähentäminen PROC GVARCLUS -menetelmällä

Instrumentoidulla tuotantolinjalla on tavallista kirjata kymmeniä antureita, jotka mittaavat päällekkäisiä fysikaalisia ilmiöitä — useita termopareja samalla vyöhykkeellä, redundantteja paineantureita, tärinäantureita, jotka seuraavat samaa akselia. Jokaisen kanavan syöttäminen ohjauskorttiin tai jatkomalliin tuhlaa seurantaresursseja ja lisää multikollineaarisuutta.

**Muuttujaklusterointi** ratkaisee tämän suoraan. `PROC GVARCLUS` jakaa numeeriset muuttujat *erillisiin* klustereihin, joissa kutakin klusteria kuvaa sen jäsenten ensimmäinen pääkomponentti. Yhdessä liikkuvat anturit päätyvät samaan klusteriin; insinööri voi sitten pitää yhden edustavan kanavan klusteria kohden.

Tässä muistikirjassa:

1. Luodaan lukemat 3-vyöhykkeiseltä linjalta tarkoituksella redundanteilla antureilla (silmukka pyytää 300 kierrosta; 100 säilytetään tässä).
2. Klusteroidaan 13 kanavaa `PROC GVARCLUS`-menetelmällä asetuksella `MAXCLUSTERS=4`.
3. Tallennetaan klusteriosoitukset tulosteaineistoon ja tehdään niistä yhteenveto.
4. Tulkitaan R²-arvot valitakseen yhden mittarin klusteria kohden jatkuvaan SPC-seurantaan.

## Vaihe 1 — Luo synteettinen anturidata

Simuloimme kolmea prosessivyöhykettä, kullakin piilevä ajuri (`zone1_a`, `zone2_a`, `zone3_a`). Loput kanavat ovat kohinaisia lineaarisia funktioita oman vyöhykkeensä ajurista, joten ne korreloivat voimakkaasti vyöhykkeen sisällä mutta ovat lähes riippumattomia vyöhykkeiden välillä. Sidomme myös sisään- ja ulostulolämpötilat vyöhykkeeseen 1 ja molemmat paineanturit vyöhykkeeseen 2, jäljitellen todellisilla linjoilla nähtyä instrumenttien välistä redundanssia. Kiinteä siemenluku tekee ajosta toistettavan. Silmukka pyytää 300 kierrosta; ilman lisenssiä moottori säilyttää ensimmäiset 100 havaintoa, mistä alla oleva NOTE kertoo.

In [1]:
TIEDOT process_sensors;
    CALL streaminit(20260531);
    TEE cycle = 1 ASTI 300;
        /* Vyöhyke 1: piilevä ajuri sekä kaksi redundanttia anturia */
        zone1_a = rand("normal", 0, 1);
        zone1_b = 0.90*zone1_a + rand("normal", 0, 0.30);
        zone1_c = 0.85*zone1_a + rand("normal", 0, 0.35);

        /* Vyöhyke 2: piilevä ajuri sekä kaksi redundanttia anturia */
        zone2_a = rand("normal", 0, 1);
        zone2_b = 0.88*zone2_a + rand("normal", 0, 0.30);
        zone2_c = 0.82*zone2_a + rand("normal", 0, 0.40);

        /* Vyöhyke 3: piilevä ajuri sekä yksi redundantti anturi */
        zone3_a = rand("normal", 0, 1);
        zone3_b = 0.90*zone3_a + rand("normal", 0, 0.30);

        /* Vyöhykkeiden väliset kanavat sidottu olemassa oleviin ajureihin */
        temp_in    = 180 + 4.0*zone1_a + rand("normal", 0, 1.5);
        temp_out   = 178 + 4.0*zone1_a + rand("normal", 0, 1.6);
        pressure_a = 2.5 + 0.60*zone2_a + rand("normal", 0, 0.20);
        pressure_b = 2.5 + 0.58*zone2_a + rand("normal", 0, 0.22);
        vibration  = 0.40 + 0.30*zone3_a + rand("normal", 0, 0.10);
        TULOSTE;
    LOPPU;
    POISTA cycle;
SUORITA;


NOTE: DATA process_sensors

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote process_sensors (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.07 seconds
  cpu   0.07 seconds


## Vaihe 2 — Klusteroi anturit

Kutsumme `PROC GVARCLUS`-menetelmää kaikille 13 kanavalle. `VAR`-lause listaa klusteroitavat numeeriset anturit. Rajoitamme osituksen neljään klusteriin asetuksella `MAXCLUSTERS=4` (odotamme suunnilleen yhtä klusteria fyysistä vyöhykettä kohden, hieman liikkumavaralla). `ODS GRAPHICS ON` yhdessä `PLOTS=ALL`-asetuksen kanssa ottaa käyttöön muuttujaklusterin dendrogrammin; moottori kirjoittaa tämän SVG-kuvan työhakemistoon sen sijaan, että piirtäisi sen suoraan näkyviin, joten alla luettava rakenne tulee tulostetusta **Variable Cluster Assignments** -taulukosta ja klusterikohtaisesta ominaisarvotaulukosta.

`OUTPUT OUT=`-lause kirjoittaa muuttuja-klusteri-osoitukset — yhdessä kunkin muuttujan R²-arvon kanssa suhteessa omaan klusteriinsa — aineistoon, jota vasten voimme ohjelmoida myöhemmin.

In [2]:
ODS GRAPHICS ON;

PROSEDUURI gvarclus TIEDOT=process_sensors maxclusters=4 PLOTS=ALL;
    MUUTTUJA zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
    TULOSTE out=clusters;
SUORITA;

ODS GRAPHICS OFF;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: ODS Graphics is ON (width=640px, height=480px, format=SVG).
NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.
NOTE: ODS Graphics is OFF.


## Vaihe 3 — Aja uudelleen SUMMARY-asetuksella

Menetelmän ajaminen toiseen kertaan `SUMMARY`-asetuksella säilyttää saman osituksen. `PLOTS=NONE` sammuttaa grafiikan tällä ajolla. Tässä ympäristössä tulostettu raportti on identtinen Vaiheen 2 kanssa — sama 13-rivinen osoitustaulukko, samat neljä klusteria ja sama ominaisarvoyhteenveto — joten tämä solu lähinnä osoittaa, että `SUMMARY`- ja `PLOTS=NONE`-asetukset jäsentyvät ja toimivat; sen ei odoteta tuovan uusia lukuja.

In [3]:
PROSEDUURI gvarclus TIEDOT=process_sensors maxclusters=4 summary PLOTS=none;
    MUUTTUJA zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
SUORITA;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.


## Vaihe 4 — Tarkastele tulosteaineistoa

`clusters`-aineistossa (`OUTPUT OUT=`) on yksi rivi jokaista anturia kohden, sisältäen sen osoitetun `Cluster`-arvon ja `RSq_Own`-arvon (anturin ja sen klusterikomponentin välinen korrelaation neliö). Korkea `RSq_Own` tarkoittaa, että anturi on hyvin edustettuna klusterissaan — ihanteellinen ehdokas *pudotettavaksi* klusterin edustajan hyväksi. Lajittelemme klusterin mukaan ja sitten R²-arvon mukaan, joten kunkin klusterin edustavin anturi on ryhmänsä kärjessä.

In [4]:
PROSEDUURI LAJITTELE TIEDOT=clusters out=clusters_ranked;
    MUKAAN Cluster LASKEVA RSq_Own;
SUORITA;

PROSEDUURI TULOSTA TIEDOT=clusters_ranked NIMIKE;
    MUUTTUJA Variable Cluster RSq_Own;
    NIMIKE Variable = "Anturikanava"
          Cluster  = "Klusteri"
          RSq_Own  = "R² oman klusterin kanssa";
SUORITA;


  Obs  Anturikanava  Klusteri   R² oman klusterin kanssa
-----  ------------  --------  -------------------------
    1  ZONE1_A              1                    0.96867
    2  ZONE1_B              1                     0.9189
    3  TEMP_IN              1                   0.903394
    4  TEMP_OUT             1                   0.889519
    5  ZONE2_A              2                    0.98364
    6  ZONE2_B              2                   0.946708
    7  PRESSURE_A           2                   0.929356
    8  PRESSURE_B           2                   0.920915
    9  ZONE2_C              2                   0.892405
   10  ZONE3_A              3                   0.977237
   11  VIBRATION            3                    0.95916
   12  ZONE3_B              3                   0.949054
   13  ZONE1_C              4                          1




NOTE: PROC SORT data=clusters

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 13 rows from clusters.
NOTE: Wrote clusters_ranked (13 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=clusters_ranked

NOTE: PROC PRINT completed: 13 observations printed, 3 variables


## Tulosten tulkinta

Ositus palauttaa suurimman osan linjan fyysisestä rakenteesta, yhdellä rehellisellä varauksella:

- **Klusteri 1** kokoaa muuttujat `zone1_a` (R²=0,969), `zone1_b` (0,919) sekä sisään- ja ulostulolämpötilat `temp_in` (0,903) ja `temp_out` (0,890) — neljä viidestä vyöhykkeen 1 piilevän signaalin ohjaamasta kanavasta. Keskimääräinen R² 0,920.
- **Klusteri 2** kokoaa kaikki kolme vyöhykkeen 2 kanavaa `zone2_a` (0,984), `zone2_b` (0,947), `zone2_c` (0,892) yhdessä kahden paineanturin `pressure_a` (0,929) ja `pressure_b` (0,921) kanssa, jotka sidoimme vyöhykkeen 2 ajuriin. Keskimääräinen R² 0,935.
- **Klusteri 3** kokoaa muuttujat `zone3_a` (0,977), `zone3_b` (0,949) ja `vibration` (0,959). Keskimääräinen R² 0,962.
- **Klusteri 4** on yksittäinen kanava: `zone1_c`, kolmas vyöhykkeen 1 anturi, erotettiin omaksi klusterikseen R²-arvolla 1,000 (yksittäinen jäsen selittää triviaalisti itsensä täydellisesti). Vaikka `zone1_c` jakaa vyöhykkeen 1 ajurin, tällä otoskoolla menetelmä arvioi sen riittävän erilliseksi `zone1_a`/lämpötilaryhmästä ansaitakseen oman klusterinsa sen sijaan, että se yhdistettäisiin Klusteriin 1.

Kolmella monikanavaisella klusterilla on kullakin keskimääräinen R² yli **0,90**, mikä vahvistaa, että yksi komponentti selittää valtaosan niiden jäsenten vaihtelusta — juuri sen redundanssin, jonka haluamme yhdistää. Yksittäinen poikkeama — `zone1_c` päätyminen omaan klusteriinsa muun vyöhykkeen 1 sijaan — on hyödyllinen muistutus siitä, että muuttujaklusterointi on data-ohjautuvaa: useammilla kierroksilla tai suuremmalla `MAXCLUSTERS`-budjetilla raja voi siirtyä, joten ositus on lähtökohta insinöörin harkinnalle, ei kiinteä totuus.

**Toimenpide linjalle.** Pidä jokaisessa monikanavaisessa klusterissa anturi, jolla on korkein `RSq_Own` (kanava, joka edustaa parhaiten klusteriaan), ja poista tai alenna muiden prioriteettia rutiininomaisesta SPC-seurannasta — esimerkiksi `zone2_a` Klusterille 2 ja `zone3_a` Klusterille 3. Käsittele `zone1_c`-yksittäistapaus tutkittavana lippuna, ei automaattisena säilytyksenä: varmista, kantaako se aidosti erillistä informaatiota vai onko se klusterointiartefakti, ennen kuin päätät seurata sitä erikseen. Vaikka tämä yksi kanava jätettäisiin sivuun, tämä supistaa 13 seurattua kanavaa kohti neljän kanavan seurantasuunnitelmaa, vähentäen hälytyskohinaa ja multikollineaarisuutta säilyttäen samalla linjan informaatiosisällön.